In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV

In [53]:
data=pd.read_csv('kidney2.csv')
data

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,...,47,6700,4.9,no,no,no,good,no,no,notckd
396,396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,...,54,7800,6.2,no,no,no,good,no,no,notckd
397,397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,...,49,6600,5.4,no,no,no,good,no,no,notckd
398,398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,...,51,7200,5.9,no,no,no,good,no,no,notckd


In [54]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    object 
 17  wc              295 non-null    obj

In [ ]:
data.isnull().sum()

id                  0
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                70
wc                105
rc                130
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64

In [56]:
data.drop("id", axis=1, inplace=True)

In [57]:
num_cols = ['age', 'bp', 'sg', 'al', 'su']

for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

In [58]:
cat_cols = ['rbc', 'pc', 'pcc', 'ba']

for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

In [59]:
for col in cat_cols:
    print(col)
    print(data[col].unique())

rbc
['normal' 'abnormal']
pc
['normal' 'abnormal']
pcc
['notpresent' 'present']
ba
['notpresent' 'present']


In [60]:
mapping = {
    'normal': 0,
    'abnormal': 1,
    'notpresent': 0,
    'present': 1
}

data.replace(mapping, inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_12728\3437697163.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace(mapping, inplace=True)


In [61]:
data.isnull().sum(
    
)

age                 0
bp                  0
sg                  0
al                  0
su                  0
rbc                 0
pc                  0
pcc                 0
ba                  0
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                70
wc                105
rc                130
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64

In [62]:
data.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [63]:
print(data['classification'].unique())

['ckd' 'ckd\t' 'notckd']


In [64]:
data['classification'] = data['classification'].str.replace('\t', '', regex=False)
data['classification'] = data['classification'].str.strip()

In [65]:
print(data['classification'].unique())

['ckd' 'notckd']


In [66]:
data['classification'] = data['classification'].map({
    'ckd': 1,
    'notckd': 0
})

In [67]:
print(data['classification'].unique())

[1 0]


In [68]:
x= data.drop('classification', axis=1)
y = data['classification']

In [69]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [70]:
x_train.shape,x_test.shape

((320, 24), (80, 24))

In [71]:
x_train = x_train.replace('\t?', np.nan)

In [72]:
# This automatically converts all text columns into 0 and 1 columns
x_train = pd.get_dummies(x_train, drop_first=True)

In [73]:
x_test_encoded = pd.get_dummies(x_test)

# 2. Align x_test columns with x_train columns
# This adds missing columns with 0s and removes extra columns
x_test_aligned = x_test_encoded.reindex(columns=x_train.columns, fill_value=0)

In [74]:
# Separate numeric columns, calculate mean, and fill missing values
numeric_cols = x_train.select_dtypes(include=['number']).columns
x_train[numeric_cols] = x_train[numeric_cols].fillna(x_train[numeric_cols].mean())


In [75]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [76]:
Rf=RandomForestClassifier(n_estimators=100,random_state=45)

In [77]:
grid = GridSearchCV(
    estimator=Rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

In [78]:
grid.fit(x_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,estimator,RandomForestC...ndom_state=45)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [79]:
print(grid.best_params_)

{'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5}


In [80]:
best_model = grid.best_estimator_

In [81]:
y_pred = best_model.predict(x_test_aligned)
print(y_pred)

[0 0 1 1 1 1 0 1 0 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 0 1 1 0 1 0 1 0 1 1 1 1
 0 1 1 0 1 1 1 1 0 1 1 1 1 1 0 0 1 0 1 1 1 1 1 1 0 0 1 1 1 0 0 0 1 0 1 1 1
 0 1 1 0 1 0]


In [82]:
a=accuracy_score(y_pred,y_test)
a

0.9875

In [83]:
print(confusion_matrix(y_test,y_pred))

[[28  0]
 [ 1 51]]


In [84]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98        28
           1       1.00      0.98      0.99        52

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



In [85]:
data.head(20)

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,0,0,0,0,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,1
1,7.0,50.0,1.020,4.0,0.0,0,0,0,0,NaN,...,38,6000,NaN,no,no,no,good,no,no,1
2,62.0,80.0,1.010,2.0,3.0,0,0,0,0,423.0,...,31,7500,NaN,no,yes,no,poor,no,yes,1
3,48.0,70.0,1.005,4.0,0.0,0,1,1,0,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,1
4,51.0,80.0,1.010,2.0,0.0,0,0,0,0,106.0,...,35,7300,4.6,no,no,no,good,no,no,1
5,60.0,90.0,1.015,3.0,0.0,0,0,0,0,74.0,...,39,7800,4.4,yes,yes,no,good,yes,no,1
6,68.0,70.0,1.010,0.0,0.0,0,0,0,0,100.0,...,36,NaN,NaN,no,no,no,good,no,no,1
7,24.0,80.0,1.015,2.0,4.0,0,1,0,0,410.0,...,44,6900,5,no,yes,no,good,yes,no,1
8,52.0,100.0,1.015,3.0,0.0,0,1,1,0,138.0,...,33,9600,4.0,yes,yes,no,good,no,yes,1
9,53.0,90.0,1.020,2.0,0.0,1,1,1,0,70.0,...,29,12100,3.7,yes,yes,no,poor,no,yes,1
